In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np
import networkx as nx

# FUNCTIONS

In [4]:
# define a function to load a pickle
def load_pickle(file_name):
    if os.path.exists(file_name):
        with open(file_name, 'rb') as handle:
            de_pickle = pickle.load(handle)
    else:
        de_pickle = None
        print("file does not exist")
    return de_pickle

# LOAD LETTERS

In [5]:
letter_dict = load_pickle(file_name = 'letter_dict.pkl')

In [6]:
word_df = pd.read_csv(filepath_or_buffer=  'words_alpha.txt', header = None, names = ['word'], dtype = str)

In [7]:
word_df['word'] = word_df['word'].astype(str)

In [8]:
word_df.head()

,word
0,a
1,aa
2,aaa
3,aah
4,aahed


In [9]:
word_df.shape

(370105, 1)

In [10]:
word_df['word'].isna().value_counts()

word
False    370105
Name: count, dtype: int64

In [11]:
word_df['lcase'] = word_df['word'].str.lower()

In [12]:
word_df['n_letters'] = word_df['word'].str.len()

In [13]:
word_df['letters_sorted'] = word_df['lcase'].map(lambda x: ''.join(sorted(x)))

In [14]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))

In [15]:
word_df['n_unique_chars'] = word_df['lcase_set'].map(lambda x: len(x))

In [16]:
word_df = word_df.loc[(word_df['n_unique_chars'] == 5) & (word_df['n_letters'] == 5), :]
word_df = word_df.sort_values(by = 'lcase').reset_index(drop = True)

In [17]:
# using the word_group, select entries
word_df = word_df.drop_duplicates(subset = ['letters_sorted'])

In [18]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars
0,abdom,abdom,5,abdmo,"{a, o, b, m, d}",5
1,abend,abend,5,abden,"{a, b, d, n, e}",5
2,abets,abets,5,abest,"{a, b, s, t, e}",5
3,abhor,abhor,5,abhor,"{a, o, b, r, h}",5
4,abide,abide,5,abdei,"{a, b, i, d, e}",5


In [19]:
word_df.shape

(5977, 6)

In [20]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


In [21]:
word_df['word_id'] = range(0, word_df.shape[0])

In [22]:
word_df.shape

(5977, 7)

In [23]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars,word_id
0,abdom,abdom,5,abdmo,"{a, o, b, m, d}",5,0
1,abend,abend,5,abden,"{a, b, d, n, e}",5,1
2,abets,abets,5,abest,"{a, b, s, t, e}",5,2
3,abhor,abhor,5,abhor,"{a, o, b, r, h}",5,3
4,abide,abide,5,abdei,"{a, b, i, d, e}",5,4


In [24]:
word_id_list = word_df['word_id'].to_numpy(dtype = np.int16)

# BYTE ENCODE WORDS

In [25]:
def byte_encode_words(word:str) -> int:
    ret = 0
    for c in sorted(word):
        alpha_index = ord(c) - ord("a")
        ret |= 1 << alpha_index
    return ret

In [26]:
word_df['word_byte'] = word_df['word'].map(byte_encode_words)

In [27]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars,word_id,word_byte
0,abdom,abdom,5,abdmo,"{a, o, b, m, d}",5,0,20491
1,abend,abend,5,abden,"{a, b, d, n, e}",5,1,8219
2,abets,abets,5,abest,"{a, b, s, t, e}",5,2,786451
3,abhor,abhor,5,abhor,"{a, o, b, r, h}",5,3,147587
4,abide,abide,5,abdei,"{a, b, i, d, e}",5,4,283


In [28]:
word_byte_list = word_df['word_byte'].tolist()

## EXAMPLES OF BYTE COMPARISONS

In [29]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [30]:
# no letters in common
w1b & w2b

0

In [31]:
# letters in common
w1b & w3b

147456

In [32]:
w1b | w2b

673975

In [33]:
# this is the same as directly above
testo = byte_encode_words('abhorcleft')
testo

673975

In [34]:
lc_be = byte_encode_words(ascii_lowercase)

In [35]:
lc_be

67108863

In [36]:
word_byte_array = np.array(word_byte_list, dtype = np.int32)

In [37]:
word_byte_to_word_dict = {wb:lcase for wb, lcase in zip(word_df['word_byte'], word_df['lcase'])}

# BUILD LEVEL 2 USING COMBINATIONS

In [59]:
def build_l2(word_byte_list:list, focal_values:set = None) -> pd.DataFrame:
        
    l2_list = np.full(shape = (1000000, 3), fill_value = -1, dtype = np.int32)
    row_index = 0
    found_values = set()
    for w1_be, w2_be in combinations(word_byte_list, 2):
        if w1_be & w2_be == 0:   
            # they share no letters in common
            l2 = w1_be | w2_be                  

            if focal_values and l2 in focal_values:
                print('here')
                l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
                row_index += 1
            
            if focal_values is None and l2 not in found_values:
                l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
                found_values.add(l2)
                row_index += 1

    # trim the data frame
    l2_list = l2_list[:row_index, :]
    l2_df = pd.DataFrame(data = l2_list, columns = ['w1', 'w2', 'l2'])

    return l2_df
    


In [60]:
l2_df = build_l2(word_byte_list=word_byte_list, focal_values=l2_set)
l2_df.shape

here
here
here
here
here
here
here
here
here
here
here
here
here
here
here
here


(16, 3)

In [81]:
def build_l3(l2_df:pd.DataFrame, focal_values:set = None) -> pd.DataFrame:
    n_columns = 5
    l3_list = np.full(shape = (100000000, n_columns), fill_value = -1, dtype = np.int32)
    start_pos = 0
    
    found_values = set()    
    
    for i_row, row in l2_df.iterrows():    
        w1b, w2b, l2 = row
        # indexer
        positional_idx = (word_byte_array & l2) == 0

        # words with different letters
        output_array_w3b = word_byte_array[positional_idx]    

        # accumulated letters
        output_array_l3 = output_array_w3b | l2

        # get the shape to build the output
        n_pairs = output_array_l3.shape[0]
        temp_array = np.zeros(shape = (n_pairs, n_columns), dtype = np.int32)

        temp_array[:, 0] = w1b
        temp_array[:, 1] = w2b    
        temp_array[:, 2] = output_array_w3b
        temp_array[:, 3] = l2  # (w1b | w2b)
        temp_array[:, 4] = output_array_l3 # (w1b & w2b)    

        if focal_values is None:
            output_array_l3_test = [x not in found_values for x in output_array_l3 ]
            temp_array = temp_array[output_array_l3_test, :]
            found_values.update(temp_array[:, 4])

        n_pairs = temp_array.shape[0]
        end_pos = start_pos + n_pairs
        l3_list[start_pos:end_pos, :] = temp_array
        start_pos = end_pos

        if i_row % 10000 == 0:
            print(i_row)  

    l3_list  = l3_list[:end_pos, :]
    print(l3_list.shape)
    l3_df = pd.DataFrame(data = l3_list, columns = ['w1b', 'w2b', 'w3b', 'l2', 'l3'])

    return l3_df

In [82]:
l3_df = build_l3(l2_df=l2_df, focal_values=True)

0
(4728, 5)


In [84]:
def build_l4(l3_df:pd.DataFrame, focal_values:set=None) -> pd.DataFrame:

    n_columns = 7
    l4_list = np.full(shape = (100000000, n_columns), fill_value = -1, dtype = np.int32)
    start_pos = 0
    
    found_values = set()
        
    for i_row, row in l3_df.iterrows():    
        w1b, w2b, w3b, l2, l3 = row

        # indexer    
        positional_idx = (word_byte_array & l3) == 0
        if positional_idx.size > 0:

            # words with different letters
            output_array_w4b = word_byte_array[positional_idx]    

            # accumulated letters
            output_array_l4 = output_array_w4b | l3

            # get the shape to build the output
            n_pairs = output_array_l4.shape[0]
            temp_array = np.zeros(shape = (n_pairs, n_columns), dtype = np.int32)

            temp_array[:, 0] = w1b
            temp_array[:, 1] = w2b    
            temp_array[:, 2] = w3b
            temp_array[:, 3] = output_array_w4b
            temp_array[:, 4] = l2
            temp_array[:, 5] = l3  # (w1b | w2b | w3b)
            temp_array[:, 6] = output_array_l4 # (w1b | w2b | w3b | w4b) l4
            
            if focal_values is None:
                output_array_l4_test = [x not in found_values for x in output_array_l4 ]
                temp_array = temp_array[output_array_l4_test, :]
                found_values.update(temp_array[:, 6])

            n_pairs = temp_array.shape[0]
            end_pos = start_pos + n_pairs
            l4_list[start_pos:end_pos, :] = temp_array
            start_pos = end_pos

        if i_row % 10000 == 0:
            print(i_row)  

    l4_list = l4_list[:start_pos, :]    
    print(l4_list.shape)
    l4_df = pd.DataFrame(data = l4_list, columns = ['w1b', 'w2b', 'w3b', 'w4b', 'l2', 'l3', 'l4'])

    return l4_df


In [85]:
l4_df = build_l4(l3_df = l3_df, focal_values = True)

0
(36172, 7)


In [69]:
# save stuff....

In [86]:
def build_l5(l4_df:pd.DataFrame, focal_values:set = None) -> pd.DataFrame:

    n_columns = 9
    l5_list = np.full(shape = (100000000, n_columns), fill_value = -1, dtype = np.int32)
    start_pos = 0
    
    found_values = set()

    for i_row, row in l4_df.iterrows():    
        w1b, w2b, w3b, w4b, l2, l3, l4 = row

        # indexer    
        positional_idx = (word_byte_array & l4) == 0
        if positional_idx.sum() > 0:

            # words with different letters
            output_array_w5b = word_byte_array[positional_idx]
            #print(positional_idx)
            #print(output_array_w5b)

            # accumulated letters
            output_array_l5 = output_array_w5b | l4

            # get the shape to build the output
            n_pairs = output_array_l5.shape[0]
            temp_array = np.zeros(shape = (n_pairs, n_columns), dtype = np.int32)

            temp_array[:, 0] = w1b
            temp_array[:, 1] = w2b    
            temp_array[:, 2] = w3b
            temp_array[:, 3] = w4b
            temp_array[:, 4] = output_array_w5b
            temp_array[:, 5] = l2  # (w1b | w2b)
            temp_array[:, 6] = l3  # (w1b | w2b | w3b)
            temp_array[:, 7] = l4  # (w1b | w2b | w3b | w4b)
            temp_array[:, 8] = output_array_l5 # (w1b | w2b | w3b | w4b | w5b)            

            if focal_values is None:
                output_array_l5_test = [x not in found_values for x in output_array_l5 ]
                temp_array = temp_array[output_array_l5_test, :]
                found_values.update(temp_array[:, 8])

            n_pairs = temp_array.shape[0]
            end_pos = start_pos + n_pairs
            l5_list[start_pos:end_pos, :] = temp_array
            start_pos = end_pos

        if i_row % 10000 == 0:
            print(i_row)  

    l5_list = l5_list[:start_pos, :]
    l5_df = pd.DataFrame(data = l5_list, columns = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5'])

    return l5_df

In [87]:
l5_df = build_l5(l4_df = l4_df, focal_values=True)

0
10000
20000
30000


In [88]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327
1,16912387,8914984,1344516,4202944,35668496,25827371,27171887,31374831,67043327
2,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327
3,16912387,8914984,1351744,4195716,35668496,25827371,27179115,31374831,67043327
4,16912387,8914984,35668496,1344516,4202944,25827371,61495867,62840383,67043327


In [90]:
l5_df.shape

(168, 9)

In [91]:
for idx in range(1, 6):
    cn = f"w{str(idx)}b"
    ncn = f"w{str(idx)}"
    l5_df[ncn] = l5_df[cn].map(word_byte_to_word_dict)

In [92]:
l5_df

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5,w1,w2,w3,w4,w5
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327,ambry,fldxt,pucks,vejoz,whing
1,16912387,8914984,1344516,4202944,35668496,25827371,27171887,31374831,67043327,ambry,fldxt,pucks,whing,vejoz
2,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327,ambry,fldxt,pungs,vejoz,whick
3,16912387,8914984,1351744,4195716,35668496,25827371,27179115,31374831,67043327,ambry,fldxt,pungs,whick,vejoz
4,16912387,8914984,35668496,1344516,4202944,25827371,61495867,62840383,67043327,ambry,fldxt,vejoz,pucks,whing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,6179,17866816,458888,35668496,4719876,17872995,18331883,54000379,58720255,flamb,pungy,hdqrs,vejoz,twick
164,6179,17866816,4719876,458888,35668496,17872995,22592871,23051759,58720255,flamb,pungy,twick,hdqrs,vejoz
165,6179,17866816,4719876,35668496,458888,17872995,22592871,58261367,58720255,flamb,pungy,twick,vejoz,hdqrs
166,6179,17866816,35668496,458888,4719876,17872995,53541491,54000379,58720255,flamb,pungy,vejoz,hdqrs,twick


In [93]:
l5_df.to_excel(excel_writer='output_words.xlsx', index = False)

In [ ]:
# START AT LEVEL 2

In [ ]:
l2_set = set(l5_df['l2'].tolist())

In [ ]:
len(l2_set)

In [ ]:
l2_test.shape

In [ ]:
l2_df.shape

In [ ]:
l2_test.shape

In [ ]:
l2_df.to_csv(path_or_buf='l2.txt', sep = '\t', index = False)
l3_df.to_csv(path_or_buf='l3.txt', sep = '\t', index = False)
l4_df.to_csv(path_or_buf='l4.txt', sep = '\t', index = False)
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)

# LOAD IN PREVIOUS OUTPUT

In [94]:
l2_df = pd.read_csv(filepath_or_buffer='l2.txt', sep = '\t')
l3_df = pd.read_csv(filepath_or_buffer='l3.txt', sep = '\t')
l4_df = pd.read_csv(filepath_or_buffer='l4.txt', sep = '\t')
l5_df = pd.read_csv(filepath_or_buffer='l5.txt', sep = '\t')

In [95]:
l5_df.shape

(11, 14)

In [96]:
l2_set = set(l5_df['l2'].unique().tolist())

In [97]:
l2_set

{6316406,
 8923326,
 9308522,
 13110331,
 13502507,
 17872995,
 25203539,
 25347591,
 25827371,
 26261601,
 29397331}

In [42]:
test_l2_df = build_l2(word_byte_list=word_byte_list, focal_values=l2_set)

In [43]:
test_l2_df.shape

(640028, 3)

check
0    640012
1        16
Name: count, dtype: int64

In [ ]:
# get max valuesa
l2_df.max(axis = 0)

In [ ]:
l2_df['l2'].astype(np.int32).max()

In [ ]:
l3_df.head()

In [ ]:
l3_df.max(axis = 0)

In [ ]:
l4_df.max(axis = 0)

In [ ]:
my_columns = l5_df.columns.tolist()[:9]

In [ ]:
l5_df[my_columns].max(axis = 1)

In [ ]:
# join to get the different word combinations

In [ ]:
l5_df.head()

In [ ]:
l4_df.head()

In [ ]:
l4_df.shape

In [ ]:
l4_df.loc[l4_df['l4'] == 27784191, ]

In [ ]:
output_list = []
for ir5, row5 in l5_df.iterrows():
    l5 = row['l5']
    l5